第五周第二天

### AutoGen AgentChat — 深入探索...

1. 多模态对话
2. 结构化输出
3. 使用 LangChain 工具
4. 团队协作（Teams）

...还有一个特别的惊喜彩蛋

In [ ]:
from io import BytesIO
import requests
from autogen_agentchat.messages import TextMessage, MultiModalMessage
from autogen_core import Image as AGImage
from PIL import Image
from dotenv import load_dotenv
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent
from autogen_core import CancellationToken
from IPython.display import display, Markdown
from pydantic import BaseModel, Field
from typing import Literal
import os
import instructor
from openai import OpenAI

load_dotenv(override=True)

### 多模态对话

In [ ]:
url = "https://edwarddonner.com/wp-content/uploads/2024/10/from-software-engineer-to-AI-DS.jpeg"

pil_image = Image.open(BytesIO(requests.get(url).content))
img = AGImage(pil_image)
img

In [ ]:
multi_modal_message = MultiModalMessage(content=["Describe the content of this image in detail", img], source="User")

In [ ]:
model_client = OpenAIChatCompletionClient(
    model="mimo-v2.5",
    base_url="https://api.xiaomimimo.com/v1",
    api_key=os.getenv("MIMO_API_KEY", "sk-ct6ct1y17ry3m9xh2rce3bbx68kbsqs19y326ym89hxw2k64"),
    model_info={
        "vision": True,
        "function_calling": True,
        "json_output": True,
        "family": "unknown",
        "structured_output": False,
    },
)

describer = AssistantAgent(
    name="description_agent",
    model_client=model_client,
    system_message="You are good at describing images",
)

response = await describer.on_messages([multi_modal_message], cancellation_token=CancellationToken())
reply = response.chat_message.content
display(Markdown(reply))

### 结构化输出！

Autogen AgentChat 让它变得非常简单。

In [ ]:
class ImageDescription(BaseModel):
    scene: str = Field(description="简述：图像的整体场景")
    message: str = Field(description="图像试图传达的核心信息")
    style: str = Field(description="图像的艺术风格")
    orientation: Literal["portrait", "landscape", "square"] = Field(description="图像的朝向")

In [ ]:
model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

describer = AssistantAgent(
    name="description_agent",
    model_client=model_client,
    system_message="You are good at describing images in detail",
    output_content_type=ImageDescription,
)

response = await describer.on_messages([multi_modal_message], cancellation_token=CancellationToken())
reply = response.chat_message.content
reply

In [ ]:
import textwrap
print(f"Scene:\n{textwrap.fill(reply.scene)}\n\n")
print(f"Message:\n{textwrap.fill(reply.message)}\n\n")
print(f"Style:\n{textwrap.fill(reply.style)}\n\n")
print(f"Orientation:\n{textwrap.fill(reply.orientation)}\n\n")

### 在 AutoGen 中使用 LangChain 工具

In [ ]:
# AutoGen 的包装器：

from autogen_ext.tools.langchain import LangChainToolAdapter

# LangChain 工具：

from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_community.agent_toolkits import FileManagementToolkit
from langchain.agents import Tool


prompt = """你的任务是找到一趟 2025 年 6 月从 JFK（纽约肯尼迪）飞往 LHR（伦敦希思罗）的直飞单程航班。
首先在网上搜索有吸引力的优惠。
然后，将所有找到的优惠信息（包括完整详情）写入一个名为 flights.md 的文件中。
最后，选择你认为最合适的一趟航班，并用简短摘要进行回复。
仅在你已将详细信息写入文件之后，才回复你选定的航班。"""


serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="当你需要搜索互联网时使用此工具")
autogen_serper = LangChainToolAdapter(langchain_serper)
autogen_tools = [autogen_serper]

langchain_file_management_tools = FileManagementToolkit(root_dir="sandbox").get_tools()
for tool in langchain_file_management_tools:
    autogen_tools.append(LangChainToolAdapter(tool))

for tool in autogen_tools:
    print(tool.name, tool.description)

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
agent = AssistantAgent(name="searcher", model_client=model_client, tools=autogen_tools, reflect_on_tool_use=True)
message = TextMessage(content=prompt, source="user")
result = await agent.on_messages([message], cancellation_token=CancellationToken())
for message in result.inner_messages:
    print(message.content)
display(Markdown(result.chat_message.content))

In [ ]:
# 现在我们需要再次调用 agent 来写入文件

message = TextMessage(content="好的，请继续", source="user")

result = await agent.on_messages([message], cancellation_token=CancellationToken())
for message in result.inner_messages:
    print(message.content)
display(Markdown(result.chat_message.content))

In [ ]:
# MiMo 搜索工具封装（修复版 v3 — 纯 MiMo 加速版）
# 优化点：
#   1. reflect_on_tool_use=False — 关掉工具后的二次 LLM 调用，省 3-5s
#   2. force_search=False — 简单问题跳过搜索，省 10-20s
#   3. max_keyword=1 — 减少搜索关键词提取时间
#   4. max_completion_tokens=256 — 减少生成时间
#   5. 超时 90s + 重试 3 次

import os
import time
import requests
from langchain.tools import BaseTool
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_core import CancellationToken
from autogen_ext.tools.langchain import LangChainToolAdapter
from IPython.display import display, Markdown

# ── 1. 定义 MiMo 搜索工具 ──
class MiMoSearchTool(BaseTool):
    name: str = "mimo_search"
    description: str = "实时网页搜索。参数 query 传用户原始问题，原样传入。"

    def _run(self, query: str) -> str:
        url = "https://api.xiaomimimo.com/v1/chat/completions"
        headers = {
            "api-key": os.getenv("MIMO_API_KEY", "sk-ct6ct1y17ry3m9xh2rce3bbx68kbsqs19y326ym89hxw2k64"),
            "Content-Type": "application/json",
        }
        payload = {
            "model": "mimo-v2.5-pro",
            "messages": [{"role": "user", "content": query}],
            "tools": [{
                "type": "web_search",
                "max_keyword": 1,        # ← 优化：3→1，减少关键词提取时间
                "force_search": False,    # ← 优化：True→False，简单问题跳过搜索
                "limit": 1,
                "user_location": {
                    "type": "approximate",
                    "country": "China",
                    "region": "Hubei",
                    "city": "Wuhan",
                },
            }],
            "max_completion_tokens": 256,  # ← 优化：512→256，减少生成时间
        }
        for attempt in range(1, 4):
            try:
                resp = requests.post(url, headers=headers, json=payload, timeout=90)
                data = resp.json()
                return data["choices"][0]["message"]["content"]
            except requests.exceptions.Timeout:
                if attempt < 3:
                    print(f"  ⏳ 第 {attempt} 次超时，重试中...")
                    time.sleep(2)
                else:
                    return "[搜索超时] MiMo API 响应超过 90 秒，请稍后再试。"
            except Exception as e:
                return f"[搜索错误] {e}"

    async def _arun(self, query: str) -> str:
        return self._run(query)

# ── 2. 桥接为 AutoGen 工具 ──
mimo_search_tool = MiMoSearchTool()
autogen_mimo_search = LangChainToolAdapter(mimo_search_tool)

# ── 3. 创建 Agent ──
model_client = OpenAIChatCompletionClient(
    model="mimo-v2.5-pro",
    base_url="https://api.xiaomimimo.com/v1",
    api_key=os.getenv("MIMO_API_KEY", "sk-ct6ct1y17ry3m9xh2rce3bbx68kbsqs19y326ym89hxw2k64"),
    model_info={
        "vision": False,
        "function_calling": True,
        "json_output": True,
        "family": "unknown",
        "structured_output": False,
    },
)

agent = AssistantAgent(
    name="mimo_searcher",
    model_client=model_client,
    tools=[autogen_mimo_search],
    system_message=(
        "你是一个搜索助手。"
        "把用户的原始文字作为 query 参数传给搜索工具，不要添加额外关键词。"
        "用中文回复。"
    ),
    reflect_on_tool_use=False,  # ← 关键优化：关掉工具后的二次 LLM 调用
)

# ── 4. 调用 ──
message = TextMessage(content="厦门今天天气", source="user")
result = await agent.on_messages([message], cancellation_token=CancellationToken())

for msg in result.inner_messages:
    print(f"[{msg.source}] {msg.content}\n")

display(Markdown(result.chat_message.content))

In [11]:
# ── 测试：直接调用工具（不经过 LLM），验证 MiMo API 是否可用 ──
print("🔍 测试 MiMo 搜索工具...")
print("=" * 50)

test_query = "厦门今天天气"
print(f"查询: {test_query}")
print("等待 MiMo API 响应（最多 90 秒）...\n")

result = mimo_search_tool.run(test_query)
print(f"结果:\n{result}")
print("\n" + "=" * 50)
print("✅ 工具测试完成" if "[搜索超时]" not in result and "[搜索错误]" not in result else "❌ 工具测试失败")

🔍 测试 MiMo 搜索工具...
查询: 厦门今天天气
等待 MiMo API 响应（最多 90 秒）...

结果:
根据最新的天气信息，厦门今天（6月7日）的天气情况如下：

*   **天气现象**：雷阵雨
*   **温度范围**：25°C 到 30°C
*   **风力风向**：西南风1级
*   **空气质量**：优

今天白天天气良好，但需注意可能出现的雷阵雨天气。建议您出行时携带雨具，以备不时之需。

✅ 工具测试完成


### 团队协作（Team interactions）

In [ ]:
# 核心 Agent 类。封装了一个 LLM 驱动的助手，可以：
# 持有 system message、tools、model_client
# 自动调用工具、返回结果
# 支持 output_content_type 做结构化输出（就是你之前遇到的问题点）
# 相当于 LangChain 里的 create_react_agent
from autogen_agentchat.agents import AssistantAgent
# 终止条件。当某个 Agent 的输出中包含指定文本时，整个团队停止运行。
# termination = TextMentionTermination("TERMINATE")
# Agent 说了 "TERMINATE" → 整个 GroupChat 结束
from autogen_agentchat.conditions import  TextMentionTermination
# 团队编排模式。多个 Agent 按轮次依次发言（轮流执行）：
from autogen_agentchat.teams import RoundRobinGroupChat
# 工具适配器。把 LangChain 的工具包装成 AutoGen 能用的格式。因为 
# AutoGen 有自己的 Tool 协议，和 LangChain 的 BaseTool 不兼容，这个 adapter 做了桥接
from autogen_ext.tools.langchain import LangChainToolAdapter
# Google 搜索 API。LangChain 内置的搜索工具封装，底层调 Serper.dev 的 API
from langchain_community.utilities import GoogleSerperAPIWrapper
# 这是 LangChain 中最简单的工具创建方式——一个轻量级的函数包装器。
from langchain.agents import Tool
from langchain.tools import BaseTool
# 1. 定义自定义工具（天气查询）
#一个典型的工具类需要：
#name：工具的唯一名称（字符串）。
#description：工具的用途说明，告诉大模型什么时候该用它。
#_run(self, \args, \\kwargs)*：同步执行逻辑，返回结果。
#_arun(self, \args, \\kwargs)*：异步执行逻辑（可选），通常直接调用 _run。
class WeatherTool(BaseTool):
    name = "weather_tool"
    description = "查询指定城市的天气情况"
    def _run(self, city: str):
        # 这里写具体逻辑，比如调用天气 API
        return f"The weather in {city} is sunny and 28°C."
    async def _arun(self, city: str):
        # 异步版本，通常直接调用 _run
        return self._run(city)
# 工具对象函数封装
serper = GoogleSerperAPIWrapper()
# 也支持直接将方法函数封装成langchain的工具类。
langchain_serper =Tool(
    name="internet_search",   # 工具名称（LLM 看到的）
    func=serper.run,          # 实际执行的函数
    description="当你需要搜索互联网时使用此工具") # 告诉 LLM 什么时候用
# 只要是典型的工具类都支持LangChainToolAdapter桥接
weather_tool = WeatherTool()
weather_tool_adapter = LangChainToolAdapter(weather_tool)

# 只要是 LangChain 提供的工具类，
# 都可以通过 LangChainToolAdapter 
# 包装后交给 AutoGen Agent 使用
autogen_serper = LangChainToolAdapter(langchain_serper)

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")


prompt = """找到一趟 2025 年 6 月从 JFK 飞往 LHR 的直飞单程航班。"""


primary_agent = AssistantAgent(
    "primary",
    model_client=model_client,
    tools=[autogen_serper],
    system_message="你是一个乐于助人的 AI 研究助手，负责寻找有吸引力的航班优惠。请吸收你收到的任何反馈意见。",
)

evaluation_agent = AssistantAgent(
    "evaluator",
    model_client=model_client,
    system_message="请提供建设性的反馈意见。当你的反馈被采纳后，回复 'APPROVE'。",
)

text_termination = TextMentionTermination("APPROVE")

# 感谢 Peter A 添加了 max_turns 参数——否则这可能会陷入无限循环...

team = RoundRobinGroupChat([primary_agent, evaluation_agent], termination_condition=text_termination, max_turns=20)


In [ ]:
result = await team.run(task=prompt)
for message in result.messages:
    print(f"{message.source}:\n{message.content}\n\n")


### 咚咚咚...

## 隆重介绍 MCP！

我们第一次接触 Anthropic 提出的模型上下文协议（Model Context Protocol）——

Autogen 让使用 MCP 工具变得和使用 LangChain 工具一样简单。

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">但是等等——Windows PC 用户有一个不小的问题</h2>
            <span style="color:#ff7800;">我有一个不太好的消息。在 Windows PC 上运行 MCP Server 存在问题；Mac 和 Linux 则没有问题。截至 2025 年 5 月 4 日，这是一个已知问题。我让 o3 配合 Deep Research 尝试寻找解决方案；它<a href="https://chatgpt.com/share/6817bbc3-3d0c-8012-9b51-631842470628">确认了该问题</a>并验证了变通方法。<br/><br/>
            这个变通方法稍微有点麻烦。就是利用 "WSL"——微软在 PC 上运行 Linux 的方案。你需要完成一些额外的设置步骤！不过这个过程很快，而且多位同学已经确认这对他们来说完全可行，之后本实验以及第六周的 MCP 实验都能正常运行。另外，WSL 实际上是在 Windows PC 上构建软件的绝佳方式。你也可以跳过这最后一个单元格，但在第六周开始时你需要回来完成这部分。<br/>
            WSL 设置说明位于 Setup 文件夹中，<a href="../setup/SETUP-WSL.md">在名为 SETUP-WSL.md 的文件中</a>。希望这只会让你短暂停留——你应该很快就能恢复正常运行。哦，这就是使用前沿技术的乐趣所在！<br/><br/>
            特别感谢同学 Kaushik R. 指出这里以及第六周都需要这个设置。谢谢 Kaushik！
            </span>
        </td>
    </tr>
</table>

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.mcp import StdioServerParams, mcp_server_tools

# 从 mcp-server-fetch 获取 fetch 工具。
fetch_mcp_server = StdioServerParams(command="uvx", args=["mcp-server-fetch"], read_timeout_seconds=30)
fetcher = await mcp_server_tools(fetch_mcp_server)

# 创建一个可以使用 fetch 工具的 agent。
model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
agent = AssistantAgent(name="fetcher", model_client=model_client, tools=fetcher, reflect_on_tool_use=True)  # type: ignore

# 让 agent 获取一个 URL 的内容并进行总结。
result = await agent.run(task="查看 edwarddonner.com 并总结你了解到的内容。用 Markdown 格式回复。")
display(Markdown(result.messages[-1].content))